In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


In [ ]:
models = ['qwen-2.5-14b',
          'qwen-2.5-7b',
          'mistral-7B-v0.3',
          'gemma-2-9b',
          'gemma-7b',
          'llama-3-8b',
          'llama-3.2-3b',
          '_qwen-2.5-14b',
          '_qwen-2.5-7b',
          '_mistral-7B-v0.3',
          '_gemma-2-9b',
          '_gemma-7b',
          '_llama-3.1-8b',
          '_llama-3-8b-med',
          '_llama-3.1-8b-bio',
          '_llama-3.2-3b']
    
datapacks = ['cities_loc', 'med_indications', 'defs']
probes = ['mean_diff', 'ttpd', 'spca', 'sawmil', 'svm']
tasks = {
    'mean_diff': 3,
    'ttpd': 3,
    'spca': 3,
    'sawmil': -1,
    'svm': -1
}

def get_path(model:str, datapack: str,  probe: str,  task:int) -> Path:
    return Path('outputs/probes') / probe / model / f'{datapack}_search_task-{str(task)}' 

def get_manifest_path(model: str, datapack: str, probe: str, task: int) -> Path:
    base_path = get_path(probe = probe, model=model, datapack=datapack, task=task)
    manifest_path = base_path / 'manifests' 
    return manifest_path

def check_layer_data(model:str, datapack: str,  probe: str,  task:int):
    try:
        cfg = json.load((open(get_path(model, datapack, probe, task) / 'config.json')))
    except FileNotFoundError:
        return [-1]
    layers = cfg['model']['layers']
    layer_range = cfg['layer_range']
    q = np.quantile(layers, q=layer_range, method='nearest')
    layers = list(range(q[0], q[1]+1))
    missing_layers = []
    for layer in layers:
        manifest_path = get_manifest_path(model, datapack, probe, task) / f'manifest_{layer}.json'
        if manifest_path.exists():
            manifest = json.load(open(manifest_path))
            if Path(manifest['paths']['metrics']).exists():
                continue
            else:
                missing_layers.append(layer)
        else:
            missing_layers.append(layer)
    return missing_layers

def check_generalization_data(model:str, datapack: str,  test_datapack: str, probe: str,  task:int):
    try:
        cfg = json.load((open(get_path(model, datapack, probe, task) / 'config.json')))
    except FileNotFoundError:
        return [-1]
    layers = cfg['model']['layers']
    layer_range = cfg['layer_range']
    q = np.quantile(layers, q=layer_range, method='nearest')
    layers = list(range(q[0], q[1]+1))
    missing_layers = []
    if (get_path(model, datapack, probe, task) / f"g_{test_datapack}").exists() == False:
        return [-1]
    else:
        for layer in layers:
            manifest_path = get_path(model, datapack, probe, task) / f"g_{test_datapack}" / "manifests"/ f"manifest_{layer}.json"
            if manifest_path.exists():
                manifest = json.load(open(manifest_path))
                if Path(manifest['paths']['metrics']).exists():
                    continue
                else:
                    missing_layers.append(layer)
            else:
                missing_layers.append(layer)
    return missing_layers


# Training Data

In [ ]:
result = []
for model in models:
    for datapack in datapacks:
        for probe in probes:
            task = tasks[probe]
            missing_layers = check_layer_data(model, datapack, probe, task)
            result.append((model, datapack, probe, task, len(missing_layers)> 0,  missing_layers))

df =  pd.DataFrame(result, columns=['model', 'datapack', 'probe', 'task', 'missing_data', 'missing_layers'])
df[df['missing_data'] == True]

## Generalization Data

In [ ]:
result = []
for model in models:
    for datapack in datapacks:
        for test_datapack in datapacks:
            for probe in probes:
                task = tasks[probe]
                missing_layers = check_generalization_data(model, datapack, test_datapack, probe, task)
                result.append((model, datapack, test_datapack, probe, task, len(missing_layers)> 0,  missing_layers))

df =  pd.DataFrame(result, columns=['model', 'datapack', 'test_datapack', 'probe', 'task', 'missing_data', 'missing_layers'])
df[(df['missing_data'] == True)]